# YOLOv10-S: análisis en test

Mismo análisis que [`04_faster_rcnn_test_analysis.ipynb`](04_faster_rcnn_test_analysis.ipynb), aplicado al baseline de YOLOv10-S (`run_name="baseline"`, `best.pt`, mejor época en validación). No repetimos la inferencia: leemos las predicciones que guarda

```bash
python scripts/evaluation/evaluate_yolo.py --split val
python scripts/evaluation/evaluate_yolo.py --split test
```

en `results/metrics/yolov10s/baseline/`, con el mismo formato que las de Faster R-CNN. El script necesita el checkpoint en `models/trained/yolov10s/baseline/weights/best.pt`. Las figuras se guardan en `results/figures/yolov10s/baseline/`.

Todo lo que se decide (el umbral de confianza para la matriz de confusión y los ejemplos) se elige en **validación**; el test solo se mide.

In [ ]:
import json

import pandas as pd
from IPython.display import display

from parasite_detection import analysis as an
from parasite_detection.paths import DEDUP_DATA, RESULTS

MODEL, RUN = "yolov10s", "baseline"
metrics_dir = RESULTS / "metrics" / MODEL / RUN
figures_dir = RESULTS / "figures" / MODEL / RUN
image_dir = DEDUP_DATA / "images" / "test"

an.set_style()
coco = {split: an.load_coco(DEDUP_DATA, split) for split in ("val", "test")}
predictions = {split: an.load_predictions(metrics_dir / f"{split}_predictions.json") for split in ("val", "test")}
metrics = {split: json.loads((metrics_dir / f"{split}_metrics.json").read_text()) for split in ("val", "test")}
per_class = {split: pd.read_csv(metrics_dir / f"{split}_per_class.csv") for split in ("val", "test")}
history = pd.read_csv(metrics_dir / "training.csv")

## 1. Métricas globales

Métricas COCO (`pycocotools`) con la salida *one-to-one* de YOLOv10 (sin NMS), confianza ≥ 0.001 y como máximo 100 detecciones por imagen: es el mismo validador con el que se eligió `best.pt` durante el entrenamiento. Faster R-CNN se evaluó con su umbral por defecto de 0.05; un umbral más bajo solo añade detecciones de muy baja confianza al final de la curva PR.

El tiempo es preprocesado (letterbox a 640), red y postprocesado con lote 1, sin leer la imagen de disco y sin las 3 primeras imágenes de calentamiento, igual que la llamada al modelo que se cronometra en Faster R-CNN. Solo es comparable si ambos se midieron en la misma máquina.

In [ ]:
rows = {
    "mAP@[.5:.95]": "map", "mAP@0.5": "map50", "mAP@0.75": "map75",
    "AP small": "map_small", "AP medium": "map_medium", "AP large": "map_large",
    "AR@1": "mar1", "AR@10": "mar10", "AR@100": "mar100",
}
summary = pd.DataFrame({split: {name: metrics[split][key] for name, key in rows.items()} for split in ("val", "test")})
summary.loc["ms / imagen"] = [metrics[s]["inference"]["mean_ms"] for s in ("val", "test")]
summary.loc["imágenes"] = [metrics[s]["images"] for s in ("val", "test")]
summary.loc["objetos"] = [metrics[s]["objects"] for s in ("val", "test")]
print("Inferencia:", metrics["test"]["inference"]["device"], "|", metrics["test"]["inference"]["hardware"],
      "|", metrics["test"]["inference"]["torch_threads"], "hilos")
summary.round(3)

In [ ]:
# comprobación: el map de validación debe coincidir con el que registró el entrenamiento para best.pt
training_val = json.loads((metrics_dir / "best_validation.json").read_text())
print(f"Val ahora: mAP {metrics['val']['map']:.4f} · mAP50 {metrics['val']['map50']:.4f}")
print(f"Val en el entrenamiento: mAP {training_val['map']:.4f} · mAP50 {training_val['map50']:.4f}")

## 2. Curvas de entrenamiento

La línea discontinua marca la época con mayor mAP@[.5:.95] COCO en validación (pesos EMA), que es el `best.pt` que evaluamos. El informe interactivo del dashboard de entrenamiento está en `results/metrics/yolov10s/baseline/report.html`.

In [ ]:
an.plot_training(history, figures_dir / "training_curves.png");

## 3. Curvas PR por especie en test

Una curva por especie a IoU 0.5 y 0.75 (los 101 puntos de recall de COCO). Cuando las dos curvas casi coinciden, las cajas están bien ajustadas: lo que falta es detectar o clasificar mejor.

In [ ]:
an.plot_pr_curves(pd.read_csv(metrics_dir / "test_pr_curves.csv"), per_class["test"],
                  title="YOLOv10-S · curvas PR en test", path=figures_dir / "pr_curves_test.png");

## 4. AP por especie: validación frente a test

In [ ]:
an.plot_ap_val_test(per_class["val"], per_class["test"], path=figures_dir / "ap_per_class_val_test.png")
per_class["val"].merge(per_class["test"], on=["category_id", "name"], suffixes=("_val", "_test")).round(3)

## 5. Umbral de confianza

El mAP no necesita umbral, pero la matriz de confusión y los ejemplos sí. Elegimos el umbral con mayor F1 en **validación** (emparejamiento por clase con IoU ≥ 0.5) y lo aplicamos tal cual a test.

In [ ]:
sweep = an.threshold_sweep(coco["val"], predictions["val"])
threshold = float(sweep.loc[sweep["f1"].idxmax(), "threshold"])
an.plot_threshold_sweep(sweep, threshold, path=figures_dir / "threshold_val.png")
print(f"Umbral elegido en validación: {threshold:.2f}")
display(sweep.round(3))
print("Mismo umbral en test:")
an.threshold_sweep(coco["test"], predictions["test"], thresholds=[threshold]).round(3)

## 6. Matriz de confusión en test

Cada detección con confianza ≥ umbral se empareja, de mayor a menor confianza, con la caja real libre de mayor IoU (≥ 0.5) de **cualquier** clase. Así se ve con qué especie se confunde cada una. La fila *Fondo* son detecciones donde no hay ningún huevo; la columna *No detectado*, huevos sin ninguna detección. El color está normalizado por fila.

In [ ]:
confusion = an.confusion_matrix(coco["test"], predictions["test"], threshold)
an.plot_confusion(confusion, threshold, path=figures_dir / "confusion_matrix_test.png")
confusion

## 7. ¿Dónde se pierde el mAP? Errores tipo TIDE

Desglose simplificado de TIDE de `mapviz` (sección 9 del notebook de la práctica), con todas las detecciones y a IoU 0.5. Cada detección errónea se asigna a un tipo:

| tipo | definición |
|---|---|
| clasificación | IoU ≥ 0.5 con un huevo de otra especie |
| localización | IoU entre 0.1 y 0.5 con algún huevo |
| duplicado | IoU ≥ 0.5 con un huevo que ya tiene una detección de mayor confianza |
| fondo | IoU < 0.1 con todos los huevos |
| no detectado | huevo sin ninguna detección que lo explique |

Un oráculo corrige solo un tipo, se recalcula el mAP@0.5 y se anota la ganancia. Importa la ganancia, no el número de casos: un error de confianza 0.06 casi no resta AP.

In [ ]:
breakdown, errors_table = an.error_breakdown(coco["test"], predictions["test"])
an.plot_error_breakdown(errors_table, breakdown["base"], path=figures_dir / "error_breakdown_test.png")
print(f"mAP@0.5 de mapviz: {breakdown['base']:.4f} | pycocotools: {metrics['test']['map50']:.4f}")
errors_table.round(4)

## 8. Ejemplos cualitativos

Con el umbral de validación. Verde: detección correcta; rojo: detección errónea con su tipo de error; ámbar discontinuo: huevo no detectado.

### 8.1 Una detección correcta por especie

In [ ]:
per_image = an.image_errors(coco["test"], predictions["test"], threshold)
correct_ids, correct_names = an.pick_correct_examples(coco["test"], per_image)
an.plot_examples(coco["test"], image_dir, per_image, correct_ids,
                 [f"{n} · imagen {i}" for n, i in zip(correct_names, correct_ids)],
                 suptitle=f"Detecciones correctas en test (confianza ≥ {threshold:.2f})",
                 path=figures_dir / "examples_correct_test.jpg");

### 8.2 Un ejemplo de cada tipo de error

Para cada tipo, el error de mayor confianza, ampliado (para *No detectado*, el huevo más grande sin ninguna detección cerca).

In [ ]:
error_ids, error_keys, error_boxes = an.pick_error_examples(per_image)
an.plot_examples(coco["test"], image_dir, per_image, error_ids,
                 [f"{an.ERROR_LABELS[k]} · imagen {i}" for k, i in zip(error_keys, error_ids)],
                 ncols=len(error_ids), focus=error_boxes,
                 suptitle=f"Tipos de error en test (confianza ≥ {threshold:.2f}, vista ampliada)",
                 path=figures_dir / "examples_errors_test.jpg");

### 8.3 Imágenes con más errores

En las imágenes con muchas cajas se omiten los textos y el título da el recuento. La tabla cuenta, por especie, los huevos sin detección correcta (incluye los que tienen una detección de otra clase o mal ajustada) y si se concentran en pocas imágenes.

In [ ]:
worst_ids, counts = an.worst_images(per_image, n=8)
an.plot_examples(coco["test"], image_dir, per_image, worst_ids,
                 [f"imagen {i} · {counts[i]} errores" for i in worst_ids],
                 suptitle=f"Imágenes de test con más errores (confianza ≥ {threshold:.2f})",
                 path=figures_dir / "examples_worst_test.jpg")
an.missed_by_class(coco["test"], per_image)

## 9. Observaciones

Pendiente: completar tras ejecutar el notebook con las predicciones de `best.pt`.